In [15]:
from pathlib import Path
import numpy as np
import pandas as pd

carpeta = Path("../data/processed")
df = pd.concat(
    [pd.read_csv(f) for f in sorted(carpeta.glob("chunks_*.csv"))],
    ignore_index=True,
)
vectores = np.load(carpeta / "embeddings.npy")

# Control: debe haber un vector por cada fragmento
assert len(df) == len(vectores), "No coinciden: vuelve a correr el notebook 02"

print("fragmentos:", len(df), "| vectores:", vectores.shape)

fragmentos: 89 | vectores: (89, 384)


In [16]:
import psycopg
from pgvector.psycopg import register_vector

conn = psycopg.connect(
    "host=localhost port=5432 dbname=vectores user=proyectobd2 password=proyectobd2",
    autocommit=True,
)
conn.execute("CREATE EXTENSION IF NOT EXISTS vector")
register_vector(conn)

conn.execute("DROP TABLE IF EXISTS documentos")
conn.execute("""
    CREATE TABLE documentos (
        id          SERIAL PRIMARY KEY,
        contenido   TEXT,
        documento   TEXT,
        categoria   TEXT,
        fuente      TEXT,
        seccion     TEXT,
        numero_item INT,
        vector      vector(384)
    )
""")
print("tabla lista")

tabla lista


In [17]:
filas = [
    (
        fila.contenido,
        fila.documento,
        fila.categoria,
        fila.fuente,
        fila.seccion,
        int(fila.numero_item),
        vectores[i],
    )
    for i, fila in enumerate(df.itertuples())
]

with conn.cursor() as cur:
    cur.executemany(
        """
        INSERT INTO documentos
            (contenido, documento, categoria, fuente, seccion, numero_item, vector)
        VALUES (%s, %s, %s, %s, %s, %s, %s)
        """,
        filas,
    )

total = conn.execute("SELECT COUNT(*) FROM documentos").fetchone()[0]
print("filas en pgvector:", total)

filas en pgvector: 89


In [ ]:
from sentence_transformers import SentenceTransformer

modelo = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")

# Índice HNSW con distancia coseno
conn.execute("DROP INDEX IF EXISTS idx_documentos_hnsw")
conn.execute("""
    CREATE INDEX idx_documentos_hnsw
    ON documentos USING hnsw (vector vector_cosine_ops)
    WITH (m = 16, ef_construction = 64)
""")

def buscar_pgvector(pregunta, k=3):
    v = modelo.encode(pregunta)
    filas = conn.execute(
        """
        SELECT contenido, 1 - (vector <=> %s) AS similitud
        FROM documentos
        ORDER BY vector <=> %s
        LIMIT %s
        """,
        (v, v, k),
    ).fetchall()
    return filas

for texto, score in buscar_pgvector("puedo consultar gratis mi historial crediticio"):
    print(f"Similitud: {score:.4f}\n{texto}\n")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1988.10it/s]


Similitud: 0.4444
Lineamientos para la prevención de operaciones no reconocidas con tarjetas, artículo 13: INVESTIGACIÓN DE RECLAMOS POR TRANSACCIONES MONETARIAS NO RECONOCIDAS Las Instituciones Supervisadas deben realizar una investigación exhaustiva de todos los reclamos por transacciones monetarias no reconocidas presentadas por los tarjetahabientes, la cual debe estar debidamente documentada y disponible en caso de ser requerida por la CNBS.

Similitud: 0.4415
Lineamientos para la prevención de operaciones no reconocidas con tarjetas, artículo 3: 20. Tarjeta de Débito: Aquella que los emisores entregan a sus clientes para que, al efectuar compras o retiros en cajeros automáticos o establecimientos afiliados, los importes de estas operaciones sean debitados directamente de una cuenta de ahorro o cuenta corriente del tarjetahabiente.

Similitud: 0.4388
Lineamientos para la prevención de operaciones no reconocidas con tarjetas, artículo 3: 19. Tarjeta de Crédito: El instrumento o medi

In [ ]:
import time

preguntas = [
    "puedo consultar gratis mi historial crediticio",
    "que debe hacer el usuario antes de firmar un contrato",
    "la institucion debe atender mis reclamos",
    "como evitar el sobreendeudamiento",
    "que informacion debe darme el banco sobre mis cuentas",
    "quien supervisa a las instituciones financieras",
    "que hago si veo un cargo en mi tarjeta que yo no hice",
    "como debe avisarme el banco de una compra sospechosa con mi tarjeta",
    "que significa tarjeta de debito",
    "cuanto plazo tienen los bancos para adecuarse a estos lineamientos",
]

vectores_preg = {p: modelo.encode(p) for p in preguntas}

def consulta_pgvector(v, k):
    return conn.execute(
        "SELECT id FROM documentos ORDER BY vector <=> %s LIMIT %s", (v, k)
    ).fetchall()

# Calentamiento: consultas que no se miden
for v in list(vectores_preg.values())[:3]:
    consulta_pgvector(v, 3)

filas_tiempos = []
for k in (3, 5, 10):
    for p, v in vectores_preg.items():
        t0 = time.perf_counter()
        consulta_pgvector(v, k)
        ms = (time.perf_counter() - t0) * 1000
        filas_tiempos.append({"base": "pgvector", "k": k, "pregunta": p, "ms": ms})

tiempos = pd.DataFrame(filas_tiempos)
tiempos.to_csv("../data/processed/tiempos_pgvector.csv", index=False)

print(tiempos.groupby("k")["ms"].agg(["min", "max", "mean"]).round(3))

      min    max   mean
k                      
3   0.682  0.815  0.739
5   0.642  0.748  0.691
10  0.653  0.782  0.703
